#### Building Manifest

This is a helper table for data preprocessing.

| participant_id | age | ec_file                       | eo_file                     |
| -------------- | --: | ----------------------------- | --------------------------- |
| sub-001        |  24 | `sub-001/...EyesClosed...edf` | `sub-001/...EyesOpen...edf` |
| sub-002        |  39 | `sub-002/...EyesClosed...edf` | `sub-002/...EyesOpen...edf` |


##### Imports and paths

In [11]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\maria\Desktop\EEG-special-course")
RAW_ROOT = PROJECT_ROOT / "data" / "ds005385"
PROCESSED_ROOT = PROJECT_ROOT / "data_processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

participants_file = RAW_ROOT / "participants.tsv"
out_file = PROCESSED_ROOT / "manifest_pre_ses1.csv"


##### Load participants metadata

In [12]:
participants = pd.read_csv(participants_file, sep="\t")
participants.head()

,participant_id,sex,age,handedness,session1,late_ses1,session2,late_ses2
0,sub-001,F,60,right,yes,0,yes,0.0
1,sub-002,M,67,right,yes,0,no,NaN
2,sub-003,F,44,right,yes,0,no,NaN
3,sub-004,F,24,right,yes,0,no,NaN
4,sub-005,F,48,right,yes,0,yes,5.0


##### Build manifest

In [13]:
cols_to_keep = [c for c in ["participant_id", "age"] if c in participants.columns]
participants_small = participants[cols_to_keep].copy()

rows = []

for ec_file in RAW_ROOT.glob("sub-*/ses-1/eeg/*EyesClosed_acq-pre*_eeg.edf"):
    participant_id = ec_file.parts[-4]
    eeg_dir = ec_file.parent

    eo_matches = list(eeg_dir.glob("*EyesOpen_acq-pre*_eeg.edf"))
    if not eo_matches:
        continue

    rows.append({
        "participant_id": participant_id,
        "eyes_closed_file": str(ec_file),
        "eyes_open_file": str(eo_matches[0]),
    })

manifest = pd.DataFrame(rows)

if manifest.empty:
    raise ValueError("No matching EyesClosed/EyesOpen pre-task EDF pairs were found.")

manifest = manifest.merge(participants_small, on="participant_id", how="left")
manifest = manifest[["participant_id", "age", "eyes_closed_file", "eyes_open_file"]]
manifest = manifest.sort_values("participant_id").reset_index(drop=True)

manifest.head()

,participant_id,age,eyes_closed_file,eyes_open_file
0,sub-001,60,C:\Users\maria\Desktop\EEG-special-course\data...,C:\Users\maria\Desktop\EEG-special-course\data...
1,sub-002,67,C:\Users\maria\Desktop\EEG-special-course\data...,C:\Users\maria\Desktop\EEG-special-course\data...
2,sub-003,44,C:\Users\maria\Desktop\EEG-special-course\data...,C:\Users\maria\Desktop\EEG-special-course\data...
3,sub-004,24,C:\Users\maria\Desktop\EEG-special-course\data...,C:\Users\maria\Desktop\EEG-special-course\data...
4,sub-005,48,C:\Users\maria\Desktop\EEG-special-course\data...,C:\Users\maria\Desktop\EEG-special-course\data...


##### Save manifest in data_processed folder

In [15]:
manifest.to_csv(out_file, index=False)
print(f"Saved manifest to: {out_file}")
print(f"Number of participants with both files: {len(manifest)}")

Saved manifest to: C:\Users\maria\Desktop\EEG-special-course\data_processed\manifest_pre_ses1.csv
Number of participants with both files: 608
